# ⏩ Sunud-sunod na Workflow ng Ahente gamit ang Microsoft Foundry (Python)

## 📋 Advanced na Tutorial sa Sunud-sunod na Proseso

Ipinapakita ng notebook na ito ang **mga pattern ng sunud-sunod na workflow** gamit ang Microsoft Agent Framework. Matututuhan mo kung paano bumuo ng sopistikadong mga multi-step processing pipeline kung saan ang mga ahente ay nagpapatupad sa isang tiyak na pagkakasunod, nagpapasa ng data at konteksto sa pagitan ng mga yugto.

> **Tala sa paglipat:** Ang sample na ito ay dati nang tumutukoy sa GitHub Models. Ang GitHub Models ay deprecated na (magtatapos sa Hulyo 2026), kaya ngayon ay gumagamit na ito ng **Microsoft Foundry** sa pamamagitan ng `FoundryChatClient`, na tumutukoy sa Azure OpenAI **Responses API**.

## 🎯 Mga Layunin sa Pag-aaral

### 🔄 **Mga Pattern ng Sunud-sunod na Proseso**
- **Linear Workflow Design**: Gumawa ng hakbang-hakbang na mga processing pipeline
- **Pamahalaan ang Daloy ng Data**: Ipasa ang impormasyon sa pagitan ng sunud-sunod na mga ahente
- **Stage-Gate Processing**: Magpatupad ng mga checkpoint at yugto ng beripikasyon
- **Pagsubaybay ng Progreso**: Subaybayan ang pagpapatupad ng workflow at mga panandaliang resulta

### 🏗️ **Arkitektura ng Enterprise Pipeline**
- **Pagmomodelo ng Negosyo**: I-mapa ang totoong mga proseso ng negosyo sa workflow ng mga ahente
- **Pagsusuri ng Kalidad**: Multi-stage validation at mga proseso ng pagsusuri
- **Pagproseso ng Dokumento**: Sunud-sunod na pagsusuri at pagbago ng dokumento
- **Produksiyon ng Nilalaman**: Mga editorial na workflow na may mga yugto ng pagsusuri at pag-apruba

### 📊 **Mga Advanced na Tampok ng Workflow**
- **Pagpapanatili ng Konteksto**: Panatilihin ang estado sa buong mga yugto ng workflow
- **Pagpapasa ng Error**: Harapin ang mga pagkabigo sa sunud-sunod na proseso
- **Pag-optimize ng Pagganap**: Mahusay na mga pattern ng sunud-sunod na pagpapatupad
- **Audit Trails**: Kumpletong pagsubaybay ng mga sunud-sunod na operasyon

## ⚙️ Mga Kinakailangan at Pagsasaayos

### 📦 **Mga Depensiya**
```bash
pip install agent-framework -U
```

### 🔑 **Kompigurasyon**

Mag-sign in gamit ang Azure CLI (`az login`) upang makapag-authenticate ang `AzureCliCredential`, pagkatapos ay itakda ang mga detalye ng iyong Microsoft Foundry project.

```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4o-mini
```

## 🏢 **Mga Use Case ng Enterprise Sequential Workflow**

### 📝 **Pipeline sa Pagproseso ng Dokumento**
```
Raw Document → Content Extraction → Analysis → Validation → Final Output
```

### 🔍 **Workflow sa Pagsusuri ng Kalidad**
```
Initial Review → Technical Validation → Compliance Check → Final Approval
```

### 📰 **Pipeline sa Produksiyon ng Nilalaman**
```
Research → Writing → Editing → Review → Publishing
```

### 💼 **Awtomasyon ng Proseso ng Negosyo**
```
Data Collection → Processing → Analysis → Report Generation → Distribution
```

## 🎨 **Mga Prinsipyo sa Disenyo ng Sunud-sunod na Workflow**

- **🔗 Linear na Pag-unlad**: Ang bawat yugto ay umaasa sa output ng naunang yugto
- **📋 Pamamahala ng Estado**: Panatilihin ang konteksto at data sa lahat ng yugto
- **🛡️ Paghawak sa Error**: Mahinahon na pamamahala ng pagkabigo sa anumang yugto
- **📊 Pagsubaybay ng Progreso**: Subaybayan ang pagkumpleto at pagganap sa bawat yugto
- **🔄 Reusabilidad ng Yugto**: Disenyuhin ang mga reusable na bahagi ng workflow

Gumawa tayo ng sopistikadong sunud-sunod na mga processing workflow! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
from agent_framework import (
    Message,
    WorkflowBuilder,
    WorkflowEvent,
    WorkflowViz,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:

import os
import base64
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
SalesAgentName = "Sales-Agent"
SalesAgentInstructions = "You are my furniture sales consultant, you can find different furniture elements from the pictures and give me a purchase suggestion"

In [ ]:
PriceAgentName = "Price-Agent"
PriceAgentInstructions = """You are a furniture pricing specialist and budget consultant. Your responsibilities include:
        1. Analyze furniture items and provide realistic price ranges based on quality, brand, and market standards
        2. Break down pricing by individual furniture pieces
        3. Provide budget-friendly alternatives and premium options
        4. Consider different price tiers (budget, mid-range, premium)
        5. Include estimated total costs for room setups
        6. Suggest where to find the best deals and shopping recommendations
        7. Factor in additional costs like delivery, assembly, and accessories
        8. Provide seasonal pricing insights and best times to buy
        Always format your response with clear price breakdowns and explanations for the pricing rationale."""

In [ ]:
QuoteAgentName = "Quote-Agent"
QuoteAgentInstructions = """You are a assistant that create a quote for furniture purchase.
        1. Create a well-structured quote document that includes:
        2. A title page with the document title, date, and client name
        3. An introduction summarizing the purpose of the document
        4. A summary section with total estimated costs and recommendations
        5. Use clear headings, bullet points, and tables for easy readability
        6. All quotes are presented in markdown form"""

In [ ]:
sales_agent = provider.as_agent(
    name=SalesAgentName,
    instructions=SalesAgentInstructions,
)

price_agent = provider.as_agent(
    name=PriceAgentName,
    instructions=PriceAgentInstructions,
)

quote_agent = provider.as_agent(
    name=QuoteAgentName,
    instructions=QuoteAgentInstructions,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=sales_agent)
    .add_edge(sales_agent, price_agent)
    .add_edge(price_agent, quote_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra (`pip install graphviz`) plus the
# graphviz system binary; if it's not available, fall back to the text strings above.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
image_path = "../imgs/home.png"
with open(image_path, "rb") as image_file:
    image_b64 = base64.b64encode(image_file.read()).decode()
image_uri = f"data:image/png;base64,{image_b64}"


In [ ]:
# Note: the original notebook used a multimodal ChatMessage with an image of a
# living room. The current Message class no longer ships TextContent/DataContent
# helpers, so this migration uses a textual description of the same scene to
# keep the lesson focused on sequential workflow mechanics.
message = Message(
    role="user",
    text=(
        "I am furnishing a modern living room and want pieces that fit a warm, "
        "inviting style: a comfortable three-seat sofa, two accent armchairs, a "
        "wooden coffee table, a TV stand, a floor lamp, and a soft area rug. "
        "Please find appropriate furniture and give the corresponding price for "
        "each piece, then produce a final purchase quote."
    ),
)

In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The final stage (quote_agent) is the only output here.
events = await workflow.run(message)
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
